## Как писать Sampler?

[Общие требования к коду](https://confluence.delta.sbrf.ru/pages/viewpage.action?pageId=12039618894)

Все sampler'ы должны находиться в папке `llm_val/sampler/` и наследоваться от родительского класса `Sampler`, который лежит в файле `llm_val/sampler/Sampler.py`. У sampler'а есть 3 атрибута для каждой выборки: `train` - обучающий набор данных; `val` - валидационный набор данных; `test` - тестовый набор данных. Доступ к атрибутам реализован через getter'ы и setter'ы. Каждый аттрибут представляет собой словарь с ключами: `X` и `y` и значениями `pd.DataFrame`:
```python
test = {'X': pd.DataFrame,
        'y': pd.DataFrame}
```
При написании собственного sampler'а предобработку необходимо делать в конструкторе класса и присваивать атрибуты через конструктор родительского класса:

```python
test = self._preproc(test)
super().__init__(test=test)
```
Почему необходимо делать так? Потому что в родительском классе в setter'ах реализована проверка на корректность словаря. Также, если в вашей задаче используется только один датасет (например, тестовый), то в родительский конструктор можно передать только его, по умолчанию остальные будут `None`.

### Структура для датасетов

Для каждого формата датасетов определена (***пока не определена***) строгая структура для корректной работы с библиотекой `llm_val`.

**MMLU**:
- ***X***:
    1. *system* - Системный промпт. `str`;
    2. *question* - Вопрос. `str`;
    3. *context* - Список с контекстами. `List[str]`;
    4. *thema* - Тематика вопроса. `str`;
    5. *answer_options* - Словарь с вариантами ответов. `Dict[str, str]`;
    6. *true_answer* - Правильный ответ. Одна латинская буква. `str`;
    7. *raw_prompt* - Весь запрос, который целиком будет передаваться в модель. `str`;
    8. *model_answer* - Распаршенный ответ модели. Одна латинская буква. `str`.
- ***y***:
    1. *metric_value* - Посчитанная метрика на конкретном вопросе. 1, если `true_answer == model_answer`, иначе 0. `Union[float, int]`.

**RuBQ**:
- ***X***:
    1. *system* - Системный промпт. `str`;
    2. *question* - Вопрос. `str`;
    3. *context* - Список с контекстами. `List[str]`;
    4. *thema* - Тематика вопроса. `str`;
    5. *true_answer* - Список правильных ответов. `List[str]`;
    6. *raw_prompt* - Весь запрос, который целиком будет передаваться в модель. `str`;
    7. *model_answer* - Распаршенный ответ модели. `str`;
- ***y***:
    1. *metric_value* - Посчитанная метрика на конкретном вопросе. Например, если нас интересует только совпадение, то 1, если `true_answer` в том или ином виде содержится в `answer`, и 0 иначе. `Union[float, int]`.

**LongAnswer (в работе)**:
- ***X***:
    1. *system* - Системный промпт. `str`;
    2. *question* - Вопрос. `str`;
    3. *context* - Список с контекстами. `List[str]`;
    4. *thema* - Тематика вопроса. `str`;
    5. *true_answer* - Список релевантных пассажей. `List[str]`;
    6. *raw_prompt* - Весь запрос, который целиком будет передаваться в модель. `str`;
    7. *model_answer* - Распаршенный ответ модели. `str`;
    8. *<название_критерия>* - n-ое кол-во столбцов с оценкой асессоров по критериям. В каждом столбце находятся словари, где ключ - id асессора, а значение - оценка по критерию. `Dict[str, float]`.
- ***y***:
    1. *<название_критерия>* - Посчитанная метрика на конкретном вопросе (В случае LongAnswer, например, по алгоритму Дэвида-Скина). `Union[float, int]`.


### Этапы написания sampler'а

0. **Смотрим и думаем**. Перед написанием препроцессинга надо понять, что должно быть в семплере и что надо достать из того, что прислали контрагенты.
    1. *Смотрим, что прислали контрагенты*.
    2. *Смотрим структуру для датасетов*. 
1. **Создаем класс**. Называем его как-нибудь осмыслленно и определяем конструктор.
2. **Пишем логику препроцессинга**. На данном этапе все индивидуально: главное, чтобы конечный вариант совпадал с структурой. Логику препроцессинга желательно выносить в отдельную функцию.
3. **Передаем предобработанные выборки в родительский класс**.

Опциональные пункты, если вы хотите добавить свой sampler в качестве примера в `llm_val`:

4. **Пишем докстринги**. Комментарии являются важной частью при написании любого кода. [Требования к комментариям (Раздел "Документация")](https://confluence.delta.sbrf.ru/pages/viewpage.action?pageId=12039618894).
5. **Пишем юнит-тесты**. Иерархия юнит-тестов повторяет иерархию библиотеки. Тесты по sampler'ам находятся в файле `unit_tests/sampler/test_sampler.py`.

## Пример с рассуждениями

Попробуем написать свой sampler, используя план выше. Допустим, что необходимо написать sampler для MMLU датасета, и контрагенты прислали только тестовую выборку.

### 0. Смотрим и думаем

#### A. Смотрим что прислали контрагенты

Читаем данные и видим три столбца `query`, `answer`, `true_answer`. В `query` лежит следующий текст:
```
'\nКонтекст:\nЧто такое ипотека и ипотечный кредит\nИпотечный кредит – долгосрочный кредит, предоставляемый банками под залог недвижимости.\n\nВыбор программы кредитования зависит от следующих факторов:\n1) Цель клиента. Пример: приобрести готовый или строящийся ОН.\n2) Условия программы кредитования. Пример: корпоративная программа для сотрудников Сбера или индивидуальные программы от застройщиков.\n3) Параметры одобрения на заявку. Пример: если нет возможности предоставить документы, подтверждающие трудовую занятость, клиенту подойдет программа «Ипотека по двум документам».\n4) Гражданство клиента. Иностранный гражданин может оформить кредит по программам: Приобретение готового жилья, Приобретение строящегося жилья, Загородная недвижимость, Строительство жилого дома, Гараж. Одобрение происходит по децентрализованной (ДЦ) технологии.\n\n\nВопрос:\nКакие объекты недвижимости можно приобрести с использованием ипотечного кредита по программе строящееся жилье?\nВарианты ответов:\nA) Жилое помещение\nB) Коммерческое помещение\nC) Гараж\nD) Офисное помещение\n\nТвой ответ:\n'
```

#### B. Смотрим структуру для датасетов

Смотрим пункт ["Структура для датасетов"](#Структура-для-датасетов) и видим, что для MMLU датасета должно быть семь колонок в `X` и одна в `y`. Видим, что данные контрагентов не совпадают со структурой, поэтому необходимо привести данные к целевому виду.

### 1. Создаем класс

Так как у нас MMLU датасет, то класс назовем `MMLUSampler`. В конструкторе пропишем только `test`, так как контрагенты передали только тестовую выборку.
```python
class MMLUSampler(Sampler):
    def __init__(self, 
                 test: pd.DataFrame):
        pass
```

### 2. Пишем логику препроцессинга

Логику препроцессинга вынесем в отдельную функцию. Получение колонок тоже декомпозируем на функции, чтобы можно было сделать `pd.DataFrame.apply()`. Для примера напишем функцию для получения столбца `question`:

```python
def _parse_question(self, text: str) -> str:
    pattern = r'Вопрос:\s*(.+?)(?=\nВарианты ответов:)'
    question_match = re.search(pattern, text, re.DOTALL)
    question = question_match.group(1) if question_match else None
    return question
```
После написания всех функций для препроцессинга можно всё это объединить в одну функцию и получить следующий код:
```python
def _preproc(self, var: pd.DataFrame) -> tp.Dict[str, pd.DataFrame]:
    X = pd.DataFrame()
    y = pd.DataFrame()
    X['system'] = None
    X['question'] = var['query'].apply(self._parse_question)
    X['context'] = var['query'].apply(self._parse_context)
    X['thema'] = None
    X['answer_options'] = var['query'].apply(self._parse_answer)
    X['true_answer'] = var['true_answer'].apply(self._parse_true_answer)
    X['raw_prompt'] = var.apply(lambda row: self._parse_all_prompt(row), axis=1)
    X['model_answer'] = var['answer'].apply(self._parse_model_answer)
    y['metric_value'] = (X['true_answer'] == X['model_answer']).astype(int)
    return {'X': X, 'y': y}
```
После написания препроцессинга желательно пробежаться глазами по обработанным данным и проверить, что вы написали все правильно🤩

### 3. Передаем предобработанные выборки в родительский класс

Дописываем наш конструктор, и семплер готов:
```python
def __init__(self, 
             test: pd.DataFrame):
    test = self._preproc(test)
    super().__init__(test=test)
```
Полную реализацию можно посмотреть в файле: `llm_val/sampler/MMLUSampler.py`

### 4. Пишем докстринги

Самая важная часть написания семплера для библиотеки - это написание документации. Подробнее посмотреть требования к документации можно [здесь (Раздел "Документация")](https://confluence.delta.sbrf.ru/pages/viewpage.action?pageId=12039618894).
Код с комментариями выглядит следующим образом:
```python
class MMLUSampler(Sampler):
    '''
    Пример реализации sampler'а для MMLU датасетов.
    '''
    def __init__(self, 
                 test: pd.DataFrame):
        '''
        test: Тестовый набор данных
        '''
        test = self._preproc(test)
        super().__init__(test=test)


    def _preproc(self, var: pd.DataFrame) -> tp.Dict[str, pd.DataFrame]:
        X = pd.DataFrame()
        y = pd.DataFrame()
        X['system'] = None
        X['question'] = var['query'].apply(self._parse_question)
        X['context'] = var['query'].apply(self._parse_context)
        X['thema'] = None
        X['answer_options'] = var['query'].apply(self._parse_answer)
        X['true_answer'] = var['true_answer'].apply(self._parse_true_answer)
        X['raw_prompt'] = var.apply(lambda row: self._parse_all_prompt(row), axis=1)
        X['model_answer'] = var['answer'].apply(self._parse_model_answer)
        y['metric_value'] = (X['true_answer'] == X['model_answer']).astype(int)
        return {'X': X, 'y': y}
    
    
    def _parse_all_prompt(self, row: pd.Series):
        '''
        Парсит весь запрос, который целиком будет передаваться в модель.
        Parameters:
            row: Строка датасета
        Return:
            Запрос в модель
        '''
        prompt = row['query']
        return prompt
    

    def _parse_question(self, text: str) -> str:
        '''
        Парсит вопрос из запроса
        Parameters:
            text: Запрос, в котором содержится вопрос
        Return:
            Текст вопроса
        '''
        pattern = r'Вопрос:\s*(.+?)(?=\nВарианты ответов:)'
        question_match = re.search(pattern, text, re.DOTALL)
        question = question_match.group(1) if question_match else None
        return question
```

### 5. Пишем юнит-тесты

Вторая самая важная часть в написании семплера для библиотеки - это написание юнит-тестов. Все файлы с юнит-тестами получают префикс test_.

Обязательные(минимальные) проверки в юнит-тестах:
1. Проверка на корректность создания sampler'а;
2. Проверка на корректность колонок в `X` и `y`;
3. Проверка на некорректность создания sampler'а.

Пример юнит-тестов можно посмотреть в `unit_tests/sampler/test_sampler.py`